## Noise-set independence

Is the MUSE profile driven by the choice of probing noise? The eighteen DEMAND recordings divide, by their role in VoiceBank-DEMAND, into the five that form its test split (out of distribution for the pretrained MUSE), the eight that form its training split, and the five that enter neither. The paper fits the per-layer slope profile independently on these subsets over the same 824 utterances and 41 SNR levels and finds that the slopes agree at r = 0.9812 and ρ = 0.9757 with the same peak layer, and that the small difference in mean slope does not track familiarity (Sec. "The Robustness-Sensitivity Tradeoff Under Noise", last paragraph). This chapter reproduces those numbers from the per-noise slope table with `se_probe.profiles.noise_set_independence`.

Runtime: under 20 seconds on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import numpy as np
import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {DATA_DIR}")
print(f"Paper tables from: {TABLES}")

In [ ]:
from se_probe.layers import (
    DEMAND_NEITHER_SPLIT,
    HELD_OUT_NOISES_OLD,
    VOICEBANK_DEMAND_TEST_NOISES,
    VOICEBANK_DEMAND_TRAIN_NOISES,
    layout,
    probed_layers,
    short_labels,
)
from se_probe.profiles import noise_set_independence

per_noise = pd.read_csv(TABLES / "snr/per_noise_beta_muse.csv", index_col=0)
per_noise = per_noise.loc[probed_layers("muse", per_noise.index)]
assert per_noise.shape == (24, 18), per_noise.shape
assert set(VOICEBANK_DEMAND_TEST_NOISES + VOICEBANK_DEMAND_TRAIN_NOISES + DEMAND_NEITHER_SPLIT) == set(per_noise.columns)
print("test five :", VOICEBANK_DEMAND_TEST_NOISES)
print("train eight:", VOICEBANK_DEMAND_TRAIN_NOISES)
print("neither   :", DEMAND_NEITHER_SPLIT)

## Test split versus training split

The profile of a noise set is the mean of its per-noise slopes, which equals fitting the mean curve pooled over that set (the fit is linear in the curve).

In [ ]:
res = noise_set_independence(per_noise, VOICEBANK_DEMAND_TEST_NOISES, VOICEBANK_DEMAND_TRAIN_NOISES)
labels = dict(zip(per_noise.index, short_labels("muse", per_noise.index)))
print(f"held-out five vs training eight: r = {res['pearson_r']:.4f} (paper 0.9812), rho = {res['spearman_rho']:.4f} (paper 0.9757)")
print(f"peak layer: {labels[res['peak_layer_a']]} vs {labels[res['peak_layer_b']]}   (paper: both Dec-L2.0)")
print(f"mean beta: held-out {res['mean_beta_a']:.4f} (paper 0.0145), training {res['mean_beta_b']:.4f} (paper 0.0136)")
assert abs(res["pearson_r"] - 0.9812) < 5e-5 and abs(res["spearman_rho"] - 0.9757) < 5e-5
assert labels[res["peak_layer_a"]] == labels[res["peak_layer_b"]] == "Dec-L2.0"
assert abs(res["mean_beta_a"] - 0.0145) < 5e-5 and abs(res["mean_beta_b"] - 0.0136) < 5e-5

neither = noise_set_independence(per_noise, VOICEBANK_DEMAND_TEST_NOISES, DEMAND_NEITHER_SPLIT)
print(f"the five in neither split: mean beta {neither['mean_beta_b']:.4f} (paper 0.0135), r vs held-out five = {neither['pearson_r']:.4f}")
assert abs(neither["mean_beta_b"] - 0.0135) < 5e-5

## Cross-check against the cluster computation

`IIIA_independence_18.csv` was produced on the cluster for two other contrasts, the old held-out five (with PCAFETER) against the remaining twelve, and the corrected five against the remaining thirteen. The same function reproduces both rows.

In [ ]:
ref = pd.read_csv(TABLES / "snr/IIIA_independence_18.csv").set_index("contrast")
all18 = list(per_noise.columns)
old_rest = [n for n in all18 if n not in HELD_OUT_NOISES_OLD and n != "SCAFE"]
new_rest = [n for n in all18 if n not in VOICEBANK_DEMAND_TEST_NOISES]
for tag, held, rest in [("old5_vs_12", HELD_OUT_NOISES_OLD, old_rest), ("new5_vs_13", VOICEBANK_DEMAND_TEST_NOISES, new_rest)]:
    r = noise_set_independence(per_noise, held, rest)
    print(f"{tag}: r = {r['pearson_r']:.4f} (cluster {ref.loc[tag, 'pearson_r']:.4f}), rho = {r['spearman_rho']:.4f} (cluster {ref.loc[tag, 'spearman_rho']:.4f})")
    assert abs(r["pearson_r"] - ref.loc[tag, "pearson_r"]) < 1e-9
    assert abs(r["spearman_rho"] - ref.loc[tag, "spearman_rho"]) < 1e-9
    assert abs(r["mean_beta_a"] - ref.loc[tag, "mean_beta_held"]) < 1e-12

## The three profiles and the per-noise slopes

In [ ]:
import matplotlib.pyplot as plt

from se_probe.plotting import OKABE_ITO, depth_axis

lay = layout("muse")
x = np.arange(24)
fig, ax = plt.subplots(figsize=(8, 3.4))
depth_axis(ax, lay)
for name, cols, color, mk in [("VB-DEMAND test (5)", VOICEBANK_DEMAND_TEST_NOISES, OKABE_ITO["vermillion"], "o"),
                               ("VB-DEMAND train (8)", VOICEBANK_DEMAND_TRAIN_NOISES, OKABE_ITO["blue"], "s"),
                               ("neither split (5)", DEMAND_NEITHER_SPLIT, OKABE_ITO["green"], "^")]:
    ax.plot(x, per_noise[cols].mean(axis=1), marker=mk, ms=3.5, lw=1.2, color=color, label=name)
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title("MUSE slope profile fitted on three disjoint noise sets")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout();

In [ ]:
groups = [("test", VOICEBANK_DEMAND_TEST_NOISES), ("train", VOICEBANK_DEMAND_TRAIN_NOISES), ("neither", DEMAND_NEITHER_SPLIT)]
cols = [c for _, g in groups for c in g]
fig, ax = plt.subplots(figsize=(8.5, 5))
im = ax.imshow(per_noise[cols].T.to_numpy(), aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols, fontsize=8)
ax.set_xticks(lay["ticks"])
ax.set_xticklabels(lay["ticklabels"], rotation=30, ha="right")
pos = 0
for name, g in groups:
    ax.axhline(pos - 0.5, color="w", lw=1.2)
    ax.text(24.3, pos + len(g) / 2 - 0.5, name, va="center", fontsize=8)
    pos += len(g)
ax.set_xlim(-0.5, 23.5)
fig.colorbar(im, ax=ax, label=r"per-noise $\beta_\ell$ (CKA/dB)")
ax.set_title("Per-noise slope profiles, grouped by VoiceBank-DEMAND split")
plt.tight_layout();